In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [4]:
df = pd.read_csv('/content/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())

Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [5]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))

Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


## **Task 1 — Multi-Series Line Chart with Highlight**

In [18]:
# Filter Asian countries
asia_df = df[df['Region'] == 'Asia']

# Country to highlight
highlight_country = 'India'

# Base line chart
fig = px.line(
    asia_df,
    x='Year',
    y='CO2_Mt',
    color='Country',
    line_group='Country'
)

# Update layout with title and height
fig.update_layout(
    title='CO2 Emissions by Country',
    height=500,
    yaxis_title='CO₂ Emissions (Million Tonnes)' # Set the y-axis title for display
)

color_map = {c: '#2E75B6' if c == highlight_country else '#DDDDDD' for c in asia_df['Country'].unique()}

fig = px.line(asia_df, x='Year', y='CO2_Mt', color='Country',
              color_discrete_map=color_map,
              labels={'CO2_Mt': 'CO2 Emissions (Mt)', 'Year': ''})

fig.update_traces(
    line=dict(width=1.5),
    showlegend=False
    )

fig.update_traces(
    line=dict(width=3),
    selector=dict(name=highlight_country)
)

last = asia_df.loc[(asia_df['Country'] == highlight_country) & (asia_df['Year'] == asia_df['Year'].max())]

fig.add_annotation(
    x=last['Year'].values[0], y=last['CO2_Mt'].values[0],
    text=f'<b>{highlight_country}</b>', showarrow=False,
    xanchor='left', xshift=6,
    font=dict(color='#2E75B6', size=12, family='Arial')
)

fig.update_layout(
    title="China's CO2 emissions tripled 2000–2022 — no other country comes close",
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    yaxis=dict(gridcolor='#EEEEEE', title='CO2 Emissions (Mt)'),
    xaxis=dict(showgrid=False, title=''),
    margin=dict(l=60, r=80, t=55, b=40)
)

fig.show()

## **Task 2 — Slopegraph: Regional Change 2000 vs 2022**

In [21]:
selected = ['China', 'India', 'United States', 'Germany', 'Russia', 'South Korea']

sg = df.loc[
    df['Country'].isin(selected) & df['Year'].isin([2000, 2022])
].copy()

sg = sg.sort_values(['Country', 'Year'])

# Compute change
changes = sg.groupby('Country')['CO2_Mt'].agg(['first', 'last'])
changes['delta'] = changes['last'] - changes['first']

# Color logic (increase vs decrease)
color_map = {
    c: '#d62728' if row['delta'] > 0 else '#2ca02c'
    for c, row in changes.iterrows()
}

fig = px.line(
    sg,
    x='Year',
    y='CO2_Mt',
    color='Country',
    markers=True,
    color_discrete_map=color_map
)

for country in selected:
    d = sg.loc[sg['Country'] == country]

    start_val = d.iloc[0]['CO2_Mt']
    end_val = d.iloc[1]['CO2_Mt']

    fig.update_traces(
        selector=dict(name=country),
        mode='lines+markers+text',
        text=[
            f'{start_val:,.0f}',
            f'{country}<br>{end_val:,.0f}'
        ],
        textposition=['middle left', 'middle right'],
        textfont=dict(
            size=11,
            color=color_map[country]
        ),
        line=dict(width=3),
        showlegend=False
    )

fig.update_layout(
    title='CO₂ Emissions Change (2000 → 2022)<br><sup>Red = Increase | Green = Decrease</sup>',

    xaxis=dict(
        tickvals=[2000, 2022],
        ticktext=['2000', '2022'],
        showgrid=False,
        range=[1995, 2027]
    ),

    yaxis=dict(
        showgrid=False,
        showticklabels=False,
        title=''
    ),

    plot_bgcolor='white',
    paper_bgcolor='white',

    font=dict(
        family='Arial',
        size=12
    ),

    margin=dict(l=80, r=140, t=70, b=40),
    height=650,
    width=950
)

fig.show()